In [ ]:
# == Imports ==

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset
from IPython.display import Audio, display
from dotenv import load_dotenv
import jiwer
import platform

load_dotenv()

---

# Load dataset

In [ ]:
# == Load dataset in stream mode ==

print(f"Loading LibriSpeech dataset stream...")
dataset = load_dataset("openslr/librispeech_asr", "clean", split="test", streaming=True)

In [ ]:
# == Grab first sample ==
sample = next(iter(dataset))
audio_array = sample["audio"]["array"]
sample_rate = sample["audio"]["sampling_rate"]
ground_truth_text = sample['text']

print(f"Sample Rate: {sample_rate} Hz")
print(f"Ground truth text: {ground_truth_text}")

In [ ]:
# == Plot the clean waveform ==
plt.rcParams['figure.figsize'] = (12,5)
librosa.display.waveshow(audio_array, sr=sample_rate)
plt.title('Clean Speech Waveform - Sample 1')
plt.show()

In [ ]:
# == Listen to the audio ==
Audio(audio_array, rate=sample_rate)

---

# Gaussian White Noise Injection

In [ ]:
# == Generate noise and combine with audio sample ==

# first get the size of the audio array
n_array = len(audio_array)
# print(f"Length of audio: {n_array / sample_rate}s")

# generate random noise
noise_raw = (np.random.rand(n_array) * 2.0 ) -1.0

print(f"Raw Noise info: \n\tmin:  {round(float(np.min(noise_raw)), 8)} \n\tmax:  {round(float(np.max(noise_raw)), 8)} \n\tmean: {round(float(np.mean(noise_raw)), 8)}")

# calculate RMS of audio signal and noise
rms_audio_array = np.sqrt(np.sum(np.square(audio_array)))
rms_noise_raw = np.sqrt(np.sum(np.square(noise_raw)))

snr_noise = 10  # 10 dB
rms_noise = rms_audio_array/(10**(snr_noise/20))

k = rms_noise / rms_noise_raw
noise_scaled = k * noise_raw

audio_array_with_noise = audio_array + noise_scaled
rms_audio_array_with_noise = np.sqrt(np.sum(np.square(audio_array_with_noise)))

print(f"RMS audio:              {rms_audio_array}")
print(f"RMS noise raw:          {rms_noise_raw}")
print(f"RMS noise in log scale: {rms_noise}")
print(f"Scaling factor k:       {k}")
print(f"Scaled Noise info: \n\tmin:  {round(float(np.min(noise_scaled)), 8)} \n\tmax:  {round(float(np.max(noise_scaled)), 8)} \n\tmean: {round(float(np.mean(noise_scaled)), 8)}")
print(f"\nRMS audio with noise: {rms_audio_array_with_noise}")

Audio(audio_array_with_noise, rate=sample_rate)

In [ ]:
# == fct: add_white_noise ==

def add_white_noise(signal, target_snr_db):
    """
    Inject additive white Gaussian noise scaled to a precise target SNR (dB)
    """

    # Calculate RMS of clean signal
    rms_signal = np.sqrt(np.sum(np.square(signal)))

    # Calculate the required RMS of noise based on target SNR in dB
    rms_noise = rms_signal / (10**(target_snr_db/20))

    # Generate noise and scale it
    noise_raw = (np.random.rand(len(signal))* 2.0) -1.0
    rms_noise_raw = np.sqrt(np.sum(np.square(noise_raw)))

    # Scale the noise to match the target noise RMS
    noise_scaled = noise_raw * (rms_noise / rms_noise_raw)

    signal_with_noise = signal + noise_scaled

    if np.max(np.abs(signal_with_noise))>1.0:
        print("Warning: Mixed audio exceeds 0 dBFS. Normalizing to prevent clipping.")
        signal_with_noise = signal_with_noise / np.max(np.abs(signal_with_noise))

    return signal_with_noise


## Test for different white noise levels

In [ ]:
test_signal_0dB = add_white_noise(audio_array, 0)
librosa.display.waveshow(test_signal_0dB, sr=sample_rate, alpha=0.5, label='noisy')
librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
plt.title(f"Audio Waveform Overlay ({0} dB SNR White Noise)")
plt.legend()
plt.show()

Audio(test_signal_0dB, rate=sample_rate)

In [ ]:
test_signal_3dB = add_white_noise(audio_array, 3)
librosa.display.waveshow(test_signal_3dB, sr=sample_rate, alpha=0.5, label='noisy')
librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
plt.title(f"Audio Waveform Overlay ({3} dB SNR White Noise)")
plt.legend()
plt.show()
Audio(test_signal_3dB, rate=sample_rate)


In [ ]:
test_signal_6dB = add_white_noise(audio_array, 6)
librosa.display.waveshow(test_signal_6dB, sr=sample_rate, alpha=0.5, label='noisy')
librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
plt.title(f"Audio Waveform Overlay ({6} dB SNR White Noise)")
plt.legend()
plt.show()
Audio(test_signal_6dB, rate=sample_rate)


In [ ]:
test_signal_9dB = add_white_noise(audio_array, 9)
librosa.display.waveshow(test_signal_9dB, sr=sample_rate, alpha=0.5, label='noisy')
librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
plt.title(f"Audio Waveform Overlay ({9} dB SNR White Noise)")
plt.legend()
plt.show()
Audio(test_signal_9dB, rate=sample_rate)

In [ ]:
test_signal_12dB = add_white_noise(audio_array, 12)
librosa.display.waveshow(test_signal_12dB, sr=sample_rate, alpha=0.5, label='noisy')
librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
plt.title(f"Audio Waveform Overlay ({12} dB SNR White Noise)")
plt.legend()
plt.show()
Audio(test_signal_12dB, rate=sample_rate)

In [ ]:
test_signal_20dB = add_white_noise(audio_array, 20)
librosa.display.waveshow(test_signal_20dB, sr=sample_rate, alpha=0.5, label='noisy')
librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
plt.title(f"Audio Waveform Overlay ({20} dB SNR White Noise)")
plt.legend()
plt.show()
Audio(test_signal_20dB, rate=sample_rate)

In [ ]:
test_signal_neg3dB = add_white_noise(audio_array, -3)
librosa.display.waveshow(test_signal_neg3dB, sr=sample_rate, alpha=0.5, label='noisy')
librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
plt.title(f"Audio Waveform Overlay ({-3} dB SNR White Noise)")
plt.legend()
plt.show()
Audio(test_signal_neg3dB, rate=sample_rate)


In [ ]:
test_signal_neg6dB = add_white_noise(audio_array, -6)
librosa.display.waveshow(test_signal_neg6dB, sr=sample_rate, alpha=0.5, label='noisy')
librosa.display.waveshow(audio_array, sr=sample_rate, color='r', alpha=0.3, label='clean')
plt.title(f"Audio Waveform Overlay ({-6} dB SNR White Noise)")
plt.legend()
plt.show()
Audio(test_signal_neg6dB, rate=sample_rate)

---

# Transcriber Engine

In [ ]:
# 1 Hardware detection
IS_APPLE_SILICON = platform.system() == "Darwin" and platform.processor() == "arm"

In [ ]:
# == setup classifier pieline ==
def load_transcriber_engine():
    """
    Automatically detect if Apple Silicon and tries to load mlx_whisper if available
    If not available, falls back to HuggingFace pipeline and setup device accordingly ("cuda", "mps", "cpu")

    Needs GLOBAL VAR: IS_APPLE_SILICON  to be set True or False
    :return:
        if MLX is used
    """

    if IS_APPLE_SILICON:
        try:
            import mlx_whisper
            print("Apple Silicon Detected. Initializing MLX.")

            def mlx_transcriber(audio_array):
                # Using MLX community pre-converted weights of the turbo model
                result = mlx_whisper.transcribe(audio_array, path_or_hf_repo="mlx-community/whisper-large-v3-turbo")
                return {"text": result["text"]}
            return mlx_transcriber

        except ImportError:
            print(f"Apple Silicon detected but 'mlx_whisper' not installed. Falling back to PyTorch MPS.")

    # 2 Standard Production/Linux Fallback
    from transformers import pipeline, AutoTokenizer, GenerationConfig
    import torch

    print("Standard architecture detected. Initializing Hugging Face pipeline.")
    device = "cuda" if torch.cuda.is_available() else "mps" #("mps" if IS_APPLE_SILICON else "cpu")

    model_id = "openai/whisper-large-v3-turbo"
    tokenizer = AutoTokenizer.from_pretrained(model_id, clean_up_tokenization_spaces=True)

    whisper_config = GenerationConfig.from_pretrained(model_id)
    whisper_config.task = "transcribe"
    whisper_config.language = "en"
    # whisper_config.num_beams = 1

    whisper_pipe = pipeline(
        "automatic-speech-recognition",
        model=model_id,
        tokenizer=tokenizer,
        dtype=torch.float16,
        device=device
    )

    # Enclose pipeline in a wrapper function to pass the manual config along
    def hf_transcriber(audio_array):
        result = whisper_pipe(audio_array, generation_config=whisper_config)
        return result

    return hf_transcriber

---

# Metrics

In [ ]:
# == fct: normalize_text() ==
cleanup_pipeline = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.Strip(),
    # jiwer.ReduceToListOfListOfWords()
])

def normalize_text(text):
    return jiwer.RemovePunctuation()(text.lower().strip())


---
# TESTS

In [ ]:
# IS_APPLE_SILICON = platform.system() == "Darwin" and platform.processor() == "arm"
IS_APPLE_SILICON= False
transcriber = load_transcriber_engine()

In [ ]:
def run_benchmark_test():   # no noise
    iterator = iter(dataset)
    target_snr = 3  # +10dB signal-to-noise ratio to be added to the samples

    running_avg_wer_clean = 0
    running_avg_wer_noisy = 0

    for i in range(20):
        sample = next(iterator)
        audio_array = sample["audio"]["array"]
        sample_rate = sample["audio"]["sampling_rate"]
        noisy_audio_array = add_white_noise(audio_array, target_snr)
        clean_ref = cleanup_pipeline(sample["text"])
        clean_hyp = cleanup_pipeline(transcriber(audio_array)['text'])
        noisy_hyp = cleanup_pipeline(transcriber(noisy_audio_array)['text'])


        wer_clean = jiwer.wer(clean_ref, clean_hyp)
        wer_noisy = jiwer.wer(clean_ref, noisy_hyp)

        print(f"\n=== SAMPLE {i+1} ===")
        print(f"Clean ref: \n{clean_ref}\n")
        print(f"Clean hyp: \n{clean_hyp}\n")
        print(f"Noisy hyp: \n{noisy_hyp}\n")
        print(f"WER clean: {wer_clean}")
        print(f"WER noisy: {wer_noisy}")

        # update WER running avg
        k = i+1
        running_avg_wer_clean = running_avg_wer_clean*(k-1)/k + wer_clean/k
        running_avg_wer_noisy = running_avg_wer_noisy*(k-1)/k + wer_noisy/k


    print(f"Final running avg:")
    print(f"\tclean: {running_avg_wer_clean}")
    print(f"\tnoisy: {running_avg_wer_noisy}")

    return

run_benchmark_test()

In [ ]:
iterator  = iter(dataset)
for i in range(20):
    s = next(iterator)
    audio_array = s["audio"]["array"]
    sr = s["audio"]["sampling_rate"]
    ref = s["text"]
    result = transcriber(audio_array)
    print(f"Ref: {ref}")
    print(f"result: {result}")

In [ ]:
audio_array